In [1]:
import sys
print(sys.executable)

/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week2-rag/.venv/bin/python


In [2]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

In [3]:
import os
import sys
from pathlib import Path

print(sys.executable)
project_root = Path.cwd()
while project_root != project_root.parent and not (
    project_root / "utils" / "env_utils.py"
).is_file():
    project_root = project_root.parent

if not (project_root / "utils" / "env_utils.py").is_file():
    raise FileNotFoundError("Could not locate utils/env_utils.py from the notebook directory.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from utils.env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week2-rag/.venv/bin/python
Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.

Python 3.13.5 satisfies requires-python: >=3.11, <3.14
package                  | required | installed | status | path                                                                            
------------------------ | -------- | --------- | ------ | --------------------------------------------------------------------------------
jupyter                  | >=1.0.0  | 1.1.1     | ✅ OK   | …a2015/ai-certification-gen-academy/week2-rag/.venv/lib/python3.13/site-packages
langchain                | >=1.3.10 | 1.4.5     | ✅ OK   | …a2015/ai-certification-gen-academy/week2-rag/.venv/lib/python3.13/site-packages
langchain-anthropic      | >=1.4.6  | 1.7.6     | ✅ OK   | …a2015/ai-certification-gen-academy/week2-rag/.venv/lib/python3.13/site-pa

![Diagram](./images/00_Basic_RAG_Pipeline.png)

# Data Extraction

The first step in any RAG pipeline is to get the data you want to work with. Here we're building a customer-support assistant for **ShopEasy**, a multi-category e-commerce platform.

Our knowledge base lives in `shopeasy_knowledge_base.json` — 51 internal support documents across five types (`runbook`, `past_ticket`, `product_doc`, `bug_report`, `faq`) spanning five product areas (`payments`, `returns`, `shipping`, `orders`, `account`).

Each entry has a `content` string and a `metadata` object (`doc_type`, `product_area`, `priority`, `platform`, `customer_tier`, `status`, `title`). We load the JSON and wrap each entry in a LangChain `Document`, carrying the metadata along so it's available at retrieval time.

![Data Extraction](./images/01_data_extraction.png)

In [4]:
# Load the ShopEasy knowledge base from JSON and wrap each entry in a Document.
with open("mock/shopeasy_knowledge_base.json", "r") as f:
    knowledge_base_data = json.load(f)

documents = [Document(page_content=entry["content"], metadata=entry["metadata"]) for entry in knowledge_base_data]

In [5]:
len(documents) # Number of documents loaded from the ShopEasy knowledge base

51

In [6]:
# Let's inspect the metadata of one of the loaded documents.
# Each support doc carries structured metadata (doc_type, product_area, priority,
# platform, customer_tier, status, title) that we'll later use for filtering.

documents[0].page_content
# documents[0].metadata

'Handling Payment Declined Errors (PAY_DECLINED_402)\n\nSymptoms:\nCustomer reports that their payment was declined at checkout. The error page shows code PAY_DECLINED_402 with the message "Your payment could not be processed." The customer may say their card works fine on other websites.\n\nCommon Causes:\n1. The card issuer declined the transaction due to insufficient funds, spending limits, or fraud detection triggers.\n2. The billing address entered at checkout does not match the address on file with the card issuer (AVS mismatch).\n3. The card has expired or the CVV entered is incorrect.\n4. The customer\'s bank is blocking international transactions (ShopEasy processes payments through our EU-based payment gateway for some regions).\n5. The customer is using a prepaid or virtual card that does not support recurring authorizations (relevant for Subscribe & Save orders).\n\nResolution Steps:\n1. Ask the customer to verify the card details: number, expiration date, CVV, and billing 

In [7]:
# To get a better sense of the content we've loaded,
# this code block defines a helper function wrap_text to format the text and then prints the content of each document.

def wrap_text(text, width=80):
    return '\n'.join([text[i:i+width] for i in range(0, len(text), width)])

for doc in documents:
    print(wrap_text(doc.page_content))
    print("-"*100)

Handling Payment Declined Errors (PAY_DECLINED_402)

Symptoms:
Customer reports 
that their payment was declined at checkout. The error page shows code PAY_DECLI
NED_402 with the message "Your payment could not be processed." The customer may
 say their card works fine on other websites.

Common Causes:
1. The card issuer
 declined the transaction due to insufficient funds, spending limits, or fraud d
etection triggers.
2. The billing address entered at checkout does not match the
 address on file with the card issuer (AVS mismatch).
3. The card has expired or
 the CVV entered is incorrect.
4. The customer's bank is blocking international 
transactions (ShopEasy processes payments through our EU-based payment gateway f
or some regions).
5. The customer is using a prepaid or virtual card that does n
ot support recurring authorizations (relevant for Subscribe & Save orders).

Res
olution Steps:
1. Ask the customer to verify the card details: number, expiratio
n date, CVV, and billing add

# Chunk the data

Now that we have our documents, the next step is to split them into smaller chunks. This is important for a few reasons:

__Vector search efficiency__: Smaller chunks are easier to search and retrieve.

__Context window limitations__: LLMs have a limited context window, so we need to make sure the retrieved information fits within that window.

__Relevance__: Smaller chunks are more likely to be focused on a specific topic, which improves the relevance of the retrieved information.

We'll use the `RecursiveCharacterTextSplitter` to split our documents. This splitter tries to split text on a series of characters (like newlines, spaces, etc.) in a recursive manner.

- `chunk_size=500`: This sets the maximum size of each chunk to 500 characters. Our support docs are fairly short, so a smaller chunk size means the longer runbooks and tickets actually split into several focused chunks instead of staying as one big blob.

- `chunk_overlap=100`: This creates an overlap of 100 characters between consecutive chunks. This helps to ensure that we don't lose any important context at the boundaries of our chunks.

- `add_start_index=True`: This will add the starting index of the chunk in the original document to the metadata.

Importantly, the metadata from each parent document (doc_type, product_area, title, …) is copied onto **every** chunk it produces, so we never lose track of where a chunk came from.

![Chunk the Data](./images/02_chunking.png)

In [8]:
# chunk the data

# ref: https://docs.langchain.com/oss/python/integrations/splitters
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
)
split_docs = text_splitter.split_documents(documents)

In [9]:
# See what chunking did: 51 documents become a larger number of smaller chunks.
print(f"Documents in:  {len(documents)}")
print(f"Chunks out:    {len(split_docs)}")
print("-" * 100)

# Find the longest document and show how it got split (with overlap + start_index).
longest_idx = max(range(len(documents)), key=lambda i: len(documents[i].page_content))
parent = documents[longest_idx]
print(f'Longest doc: "{parent.metadata["title"]}" ({len(parent.page_content)} chars)\n')

child_chunks = [
    s for s in split_docs if s.metadata.get("title") == parent.metadata["title"]
]
print(f"It split into {len(child_chunks)} chunks:\n")
for i, chunk in enumerate(child_chunks):
    start = chunk.metadata["start_index"]
    print(f"[chunk {i} | start_index={start} | {len(chunk.page_content)} chars]")
    print(chunk.page_content)
    print("-" * 100)

Documents in:  51
Chunks out:    193
----------------------------------------------------------------------------------------------------
Longest doc: "Troubleshooting Promo Code Failures (PROMO_INVALID_100)" (2179 chars)

It split into 6 chunks:

[chunk 0 | start_index=0 | 330 chars]
Troubleshooting Promo Code Failures (PROMO_INVALID_100)

Symptoms:
Customer reports that a promo code is not working at checkout. The error shown is either PROMO_INVALID_100 ("This promo code is not valid") or PROMO_EXPIRED_101 ("This promo code has expired"). Customer may insist they received the code via email or social media.
----------------------------------------------------------------------------------------------------
[chunk 1 | start_index=332 | 494 chars]
Common Causes:
1. The promo code has expired. All codes have an expiration date visible in Admin Panel > Promotions > [Code].
2. The code has a minimum order value that hasn't been met (e.g., "SAVE20 — $20 off orders $100+").
3. The code is r

In [10]:
# split document preserves the original metadata, but adds a start_index to indicate where in the original document this chunk came from.

split_docs[2].metadata

# split_docs[2].page_content

{'doc_type': 'runbook',
 'product_area': 'payments',
 'priority': 'P1',
 'platform': 'all',
 'customer_tier': 'all',
 'status': 'active',
 'title': 'Handling Payment Declined Errors (PAY_DECLINED_402)',
 'start_index': 738}

# Indexing

Now that we have our document chunks, we need to create an index that we can search. We'll use a vector store for this, which allows us to perform semantic search on our documents.

In this block, we're setting up our vector store using Pinecone and OpenAI embeddings.

- `embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)`: This initializes the embedding model that converts our document chunks into numerical vectors. `text-embedding-3-small` produces 1536-dimensional vectors by default, but the `text-embedding-3-*` models let you shorten the output via the `dimensions` parameter — here we use **512** to match our index.

- `pc = Pinecone(api_key=...)` / `index = pc.Index(...)`: This connects to Pinecone and grabs an **existing** index. Create the index in the [Pinecone console](https://app.pinecone.io/) first, making sure its dimension is **512** (to match the `dimensions=512` embeddings) and its metric is `cosine`.

- `vector_store = PineconeVectorStore(index=index, embedding=embeddings)`: This wraps the Pinecone index with our embedding model so we can add and search documents through LangChain. We store this notebook's vectors in their own `shopeasy-basic-rag` namespace.

This expects two environment variables: `PINECONE_API_KEY` and `PINECONE_INDEX_NAME` (plus `OPENAI_API_KEY` for the embeddings).

![Chunk the Data](./images/03_indexing.png)

In [11]:
#Indexing 

#define the embeddings model
#ref: https://python.langchain.com/docs/integrations/text_embedding/
#text-embedding-3-small defaults to 1536 dims, but the text-embedding-3-* models
#support shortening the output via `dimensions`. We use 512 to match the index.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)  # 512-dim vectors


In [12]:
#connect to an existing Pinecone index
#(create/manage the index in the Pinecone console; its dimension must be 512
# to match the `dimensions=512` embeddings above, with the cosine metric)
#ref: https://python.langchain.com/docs/integrations/vectorstores/pinecone/
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(os.environ["PINECONE_INDEX_NAME"])

#define the vector store (this notebook's data lives in the "shopeasy-basic-rag" namespace)
#ref: https://python.langchain.com/docs/concepts/vectorstores/
vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="maven-test",
)

In [13]:
#Now, we add our document chunks to the vector store. (Run once)
# This process will convert each chunk into a vector and store it in the database.

#run once [Already done]
# document_ids = vector_store.add_documents(documents=split_docs)
# document_ids

In [14]:
# Let's retrieve a record from the Pinecone index by its ID to confirm it was indexed.
# Note: Pinecone is eventually consistent, so a just-added vector may take a
# moment to become fetchable.
# index.fetch(ids=[document_ids[10]], namespace="shopeasy-basic-rag")

# Retrieval

With our documents indexed, we can now perform retrieval. The goal of this step is to find the most relevant document chunks for a given customer issue.

First, we'll set up our LLM and a prompt template.

- `llm = ChatOpenAI(model="gpt-4.1-mini")`: We'll use OpenAI's `gpt-4.1-mini` model for generation.

- `template = ...`: This is the prompt template that combines the retrieved context with the customer's issue. The `{context}` and `{question}` placeholders are filled in at generation time.

![Retrieval](./images/04_retrieval.png)

In [15]:
#configure the llm
llm = ChatOpenAI(model="gpt-4.1-mini")

#set the prompt template
template = """You are a customer-support assistant for ShopEasy, an e-commerce platform.
Use the following pieces of retrieved internal knowledge-base context to help resolve the customer's issue.
If the context doesn't contain the answer, say you don't have that information rather than guessing.
Be concise and practical: state the likely cause and the next step the agent should take.

Context:
{context}

Support guidance:"""

rag_prompt_template = PromptTemplate.from_template(template)

In [16]:
# A sample customer support ticket (README Query 1).
user_question = "Customer says their order never arrived even though tracking shows delivered"

In [17]:
# Now, we'll use the vector store's similarity_search method to find the top 5 most similar chunks to the customer issue.
retrieved_docs = vector_store.similarity_search(user_question, k=5)

In [18]:
# Review the retrieved chunks. We print the metadata header so we can SEE the noise:
# semantic search will surface relevant shipping docs, but may also pull in
# returns/refund docs because the language overlaps.
for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

[past_ticket / shipping] Order ORD-88921 shows delivered but never received
Ticket #T-10234 — Customer's order ORD-88921 shows delivered but never received

Customer reported: "My order was marked as delivered two days ago but I never got it. I was home all day. The tracking says it was left at the front door but nothing was there."

Investigation: Checked tracking via FedEx — GPS coordinates of the delivery scan were 0.3 miles from the customer's address. Likely misdelivered to a nearby address by the driver.
----------------------------------------------------------------------------------------------------
[runbook / shipping] Investigating Shipping Delays (SHIP_DELAYED_301)
Investigating Shipping Delays (SHIP_DELAYED_301)

Symptoms:
Customer reports their order has not arrived by the estimated delivery date. The tracking page may show SHIP_DELAYED_301 or the tracking status has not updated in 48+ hours. Customer may be frustrated especially if they paid for expedited shipping.
----

In [19]:
# The similarity_search_with_score method returns the chunks along with their similarity scores.
vector_store.similarity_search_with_score(user_question, k=5)

[(Document(id='d73e330c-7d7a-4b32-be35-1e0e4b80f75b', metadata={'customer_tier': 'regular', 'doc_type': 'past_ticket', 'platform': 'web', 'priority': 'P1', 'product_area': 'shipping', 'start_index': 0.0, 'status': 'resolved', 'title': 'Order ORD-88921 shows delivered but never received'}, page_content='Ticket #T-10234 — Customer\'s order ORD-88921 shows delivered but never received\n\nCustomer reported: "My order was marked as delivered two days ago but I never got it. I was home all day. The tracking says it was left at the front door but nothing was there."\n\nInvestigation: Checked tracking via FedEx — GPS coordinates of the delivery scan were 0.3 miles from the customer\'s address. Likely misdelivered to a nearby address by the driver.'),
  0.685939729),
 (Document(id='7b060dc1-3991-4b22-8418-e47bfbb6113e', metadata={'customer_tier': 'all', 'doc_type': 'runbook', 'platform': 'all', 'priority': 'P1', 'product_area': 'shipping', 'start_index': 0.0, 'status': 'active', 'title': 'Inves

# Generation

The final step is to use the retrieved documents to generate an answer to the customer's issue.

We'll combine the content of the retrieved documents and use our prompt template to create a final prompt for the LLM.


![Generation](./images/05_generation.png)

In [20]:
# generate answer

docs_content = "\n\n".join([doc.page_content for doc in retrieved_docs])
prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
response = llm.invoke(prompt)

In [21]:
#generated response
response.content

'The likely cause for the missing delivery is a misdelivery, as the FedEx GPS coordinates show the package was left about 0.3 miles away from your address. The next step is to contact FedEx to track down the exact delivery location or request redelivery. You can also check with neighbors nearby. If you need further assistance, please let us know.'

In [22]:
# We can also include citations in our response by extracting the source from the metadata of the retrieved docs.
# This dataset has no "source" URL, so we cite each doc by its title and doc_type instead.

sources = [f"{doc.metadata['title']} ({doc.metadata['doc_type']})" for doc in retrieved_docs]

print(f"Sources: {sources}\n\n")
print(f'Answer: {response.content}')

Sources: ['Order ORD-88921 shows delivered but never received (past_ticket)', 'Investigating Shipping Delays (SHIP_DELAYED_301) (runbook)', 'Refund not received after 10 business days for ORD-91045 (past_ticket)', 'Tracking number TRK-FX-998834 shows no movement for 5 days (past_ticket)', 'Handling Wrong Item Received (runbook)']


Answer: The likely cause for the missing delivery is a misdelivery, as the FedEx GPS coordinates show the package was left about 0.3 miles away from your address. The next step is to contact FedEx to track down the exact delivery location or request redelivery. You can also check with neighbors nearby. If you need further assistance, please let us know.


### Langchain Retreiver

LangChain provides a `Retriever` interface, which is a more general way to retrieve documents. A vector store can be used as the backbone of a retriever, but there are other types of retrievers as well.

Here, we're creating a retriever from our vector store. We can also specify search arguments like k (the number of documents to retrieve) and search_type.

In [23]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

retrieved_docs = retriever.invoke(user_question)

for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

[past_ticket / shipping] Order ORD-88921 shows delivered but never received
Ticket #T-10234 — Customer's order ORD-88921 shows delivered but never received

Customer reported: "My order was marked as delivered two days ago but I never got it. I was home all day. The tracking says it was left at the front door but nothing was there."

Investigation: Checked tracking via FedEx — GPS coordinates of the delivery scan were 0.3 miles from the customer's address. Likely misdelivered to a nearby address by the driver.
----------------------------------------------------------------------------------------------------
[runbook / shipping] Investigating Shipping Delays (SHIP_DELAYED_301)
Investigating Shipping Delays (SHIP_DELAYED_301)

Symptoms:
Customer reports their order has not arrived by the estimated delivery date. The tracking page may show SHIP_DELAYED_301 or the tracking status has not updated in 48+ hours. Customer may be frustrated especially if they paid for expedited shipping.
----

![Overall](./images/06_put_it_together.png)

In [24]:
# Now, let's put it all together in a single function.
def generate_answer(user_question):
    #retrieve the relevant docs
    retriever = vector_store.as_retriever(search_kwargs={"k": 5}, search_type='similarity')
    retrieved_docs = retriever.invoke(user_question)

    #generate
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
    response = llm.invoke(prompt)

    return response.content

generate_answer("Customer says their order never arrived even though tracking shows delivered")

'For ticket #T-10234 (order ORD-88921), the likely cause is a misdelivery by FedEx, as the GPS coordinates show the package was left about 0.3 miles from your address. The next step is to contact FedEx to locate the package at the nearby address or initiate a claim. Please advise the customer to check with neighbors and inform them you are escalating this to FedEx for resolution.'

### More sample tickets — watch the noise

Plain semantic search handles natural language well, but when a query spans multiple product areas it casts a wide net. Run these and inspect the `[doc_type / product_area]` headers to see shipping, payments, and returns docs all mixed together — exactly the problem that metadata filtering (Notebook 2) and hybrid search (Notebook 3) will fix.

In [25]:
# Helper to put the retrieved chunks on screen before the answer, so we can SEE the noise.
# Each hit shows its rank, [doc_type / product_area], title, similarity score and a one-line
# preview, followed by a tally of which product areas the results came from.
from collections import Counter
from textwrap import shorten

def show_retrieved(user_question, k=5):
    results = vector_store.similarity_search_with_score(user_question, k=k)

    print(f'Query: "{user_question}"\n')
    for rank, (doc, score) in enumerate(results, start=1):
        m = doc.metadata
        print(f"{rank}. [{m['doc_type']} / {m['product_area']}] {m['title']}  (score: {score:.3f})")
        print(f"   {shorten(doc.page_content, width=110, placeholder=' …')}")

    areas = Counter(doc.metadata["product_area"] for doc, _ in results)
    print("\nProduct areas retrieved:", ", ".join(f"{area} ×{n}" for area, n in areas.most_common()))
    print("-" * 100)

In [26]:
# Query 2: a broad query returns broadly relevant results — returns runbook, return
# policy, return FAQ, plus past return tickets... and maybe warranty/billing noise.
query = "Customer wants to return a product they bought last month"
show_retrieved(query)
print(f"Answer: {generate_answer(query)}")

Query: "Customer wants to return a product they bought last month"

1. [runbook / returns] Processing Returns and Refunds  (score: 0.556)
   - RETURN_NOT_AS_DESCRIBED: Item doesn't match listing - RETURN_CHANGED_MIND: Customer no longer wants the …
2. [runbook / returns] Processing Returns and Refunds  (score: 0.549)
   Resolution Steps: 1. Verify the item is within the return window by checking the delivery date in Admin …
3. [past_ticket / returns] Electronics return rejected due to wrong reason code  (score: 0.546)
   Investigation: Customer initiated the return through the self-service portal and selected reason code …
4. [past_ticket / returns] Customer wants to return a Final Sale item  (score: 0.535)
   Ticket #T-10756 — Customer wants to return a Final Sale item Customer reported: "I bought a jacket on sale …
5. [past_ticket / returns] Refund not received after 10 business days for ORD-91045  (score: 0.532)
   Investigation: Checked Admin Panel > Returns > [Return ID]. The ware

In [27]:
# Query 3: an issue that spans payments + shipping + returns at once — semantic
# search casts a wide net across all three areas, making it hard to focus.
query = "Customer is upset about being charged but never receiving their product"
show_retrieved(query)
print(f"Answer: {generate_answer(query)}")

Query: "Customer is upset about being charged but never receiving their product"

1. [past_ticket / payments] Customer charged for cancelled order ORD-97654  (score: 0.591)
   Ticket #T-10890 — Customer claims they were charged for a cancelled order Customer reported: "I cancelled …
2. [past_ticket / returns] Refund not received after 10 business days for ORD-91045  (score: 0.537)
   Ticket #T-10567 — Customer's refund not received after 10 business days Customer reported: "I returned my …
3. [runbook / orders] Handling Wrong Item Received  (score: 0.535)
   Handling Wrong Item Received Symptoms: Customer reports they received an item different from what they …
4. [past_ticket / shipping] Order ORD-88921 shows delivered but never received  (score: 0.513)
   Ticket #T-10234 — Customer's order ORD-88921 shows delivered but never received Customer reported: "My order …
5. [runbook / orders] Resolving Inventory Sync Errors (INV_SYNC_ERR_700)  (score: 0.510)
   Resolution Steps: 1. Apologiz